In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 2 - WEEK 9 BAYESIAN OPTIMISATION
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function2/initial_inputs.npy")
Y = np.load("function2/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [0.69805111, 0.227]
#
# GP prediction:
# mean ≈ 0.640371
# std  ≈ 0.102884
#
# Actual:
# 0.5587667304138233
#
# Standardised residual tells us how surprising the realised
# result was relative to the GP's own predictive uncertainty.
# ------------------------------------------------------------

week8_pred_mean = 0.640371
week8_pred_std = 0.102884
week8_actual = 0.5587667304138233

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement function
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Dense global search
# ------------------------------------------------------------
#
# Function 2 is only 2D, so use a dense grid over the entire
# domain rather than relying primarily on random sampling.
# ------------------------------------------------------------

axis = np.linspace(0, 1, 501)

g1, g2 = np.meshgrid(
    axis,
    axis
)

candidates = np.column_stack([
    g1.ravel(),
    g2.ravel()
])

# Remove candidates too close to previously queried points

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 6. GP prediction
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 7. Primary Expected Improvement
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 8. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 9. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 10. UCB exploration/exploitation diagnostic
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (18, 2)
Y shape: (18,)

Current best:
[0.711177 1.      ] -> 0.679662733033218

Y range:
min = -0.06562362443733738
max = 0.679662733033218
std = 0.24507765071934565

WEEK 8 CALIBRATION CHECK
Predicted mean: 0.640371
Predicted std : 0.102884
Actual        : 0.5587667304138233

Prediction error:
-0.08160426958617673

Error / predicted std:
-0.793167738289498


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1**2 * Matern(length_scale=[0.0564, 2], nu=2.5) + WhiteKernel(noise_level=0.0565)

ARD lengthscales:
[0.05639283 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.97257682 0.02742318]

Candidates after duplicate filtering:
249762

PRIMARY EI
candidate = [0.958 0.   ]
mean = 0.3679294438275411
std = 0.2403892693429259
EI = 0.01101945032213631

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.958 0.   ] 
 mean = 0.367929 
 std = 0.240389 
 EI = 0.01101945 

xi = 2.450777e-03 
 candidate = [0.958 0.   ] 
 mean = 0.367929 
 std = 0.240389 
 EI = 0.010783 

xi = 1.225388e-02 
 candidate = [0.958 0.   ] 
 mean = 0.367929 
 std = 0.240389 
 EI = 0.00987907 

xi = 2.450777e-02 
 candidate = [0.96 0.  ] 
 mean = 0.366082 
 std = 0.241406 
 EI = 0.00884229 


HIGHEST PREDICTED MEAN
candidate = [0.704 0.916]
mean = 0.6113955789659442
std = 0.06564741959158481

UCB DIAGNOSTICS

beta=0.1 
 candidate = [0.704 0.946] 
 mean = 0.611394 
 std = 0.066 
 UCB = 0.617994

In [2]:
# ============================================================
# FINAL FUNCTION 2 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 calibration error was approximately -0.79 predictive
# standard deviations, so the GP remains reasonably calibrated
# locally.
#
# Global EI is rejected because it moves to [0.958, 0.0] with
# a much lower predicted mean and very high uncertainty.
#
# Highest mean and all UCB settings remain close to the known
# high-performing region around x1 ~ 0.70 and high x2.
#
# beta = 0.5 provides a moderate exploration-exploitation
# compromise without allowing uncertainty to dominate.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week9_candidate = candidates[final_idx]

print("Week 9 Function 2 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 2 candidate:
[0.704 0.982]

Predicted mean:
0.6111988921710515

Predicted std:
0.0665952341819297

UCB:
0.6444965092620163

Portal format:
0.704000-0.982000
